# MAMMQA: Multi-Agent Multimodal Question Answering
## Complete Implementation on MultimodalQA Dataset
### Based on: *Rethinking Information Synthesis in Multimodal QA — A Multi-Agent Perspective* (IJCNLP-AACL 2025)

---

## Pipeline Architecture

```
INPUT: Question + Text Passage + Table (markdown) + Image (URL)
         |
┌────────────────────────────────────────────────┐
│  STAGE 1 — Modality Expert Agents (3 agents)   │
│  Each agent sees ONE modality + question        │
│  Task: EXTRACT insights, do NOT answer yet      │
│  Outputs: text_insight, table_insight,          │
│           image_insight                         │
└────────────────────────────────────────────────┘
         |
┌────────────────────────────────────────────────┐
│  STAGE 2 — Cross-Modal Synthesis (3 agents)    │
│  Each anchored on ONE modality's insight +      │
│  receives raw data from other two modalities    │
│  Task: Cross-modal reasoning → candidate answer │
│  Outputs: text_cross, table_cross, image_cross  │
└────────────────────────────────────────────────┘
         |
┌────────────────────────────────────────────────┐
│  STAGE 3 — Aggregator Agent (1 agent)          │
│  Blind to raw inputs — sees only Stage 2 outputs│
│  Task: Resolve conflicts → final grounded answer│
└────────────────────────────────────────────────┘
         |
      FINAL ANSWER
```

---

## Setup Required
1. **Enable Internet** in Kaggle: Notebook Settings → Internet → On
2. **Add OpenAI API Key**: Add-ons → Secrets → Add New Secret → Name: `OPENAI_API_KEY`
3. **GPU not needed** — this runs on CPU using the OpenAI API

**Estimated cost**: ~$0.05–0.15 for 5 examples with gpt-4o-mini

## Cell 1: Install Dependencies

In [ ]:
%%capture
!pip install openai python-dotenv pandas tabulate requests Pillow

## Cell 2: Imports and API Key Setup

In [ ]:
import os
import json
import re
import base64
import io
import csv
import gzip
import shutil
import urllib.request
import pandas as pd
import requests
from collections import Counter
from openai import OpenAI

print("✅ Imports successful")

# ── Load API key ──────────────────────────────────────────────────────────────
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    OPENAI_API_KEY = secrets.get_secret("OPENAI_API_KEY")
    print("✅ API key loaded from Kaggle secrets")
except Exception as e:
    # ── Fallback: paste your key here for testing only ──
    OPENAI_API_KEY = "sk-YOUR-KEY-HERE"
    print(f"⚠️  Kaggle secret not found ({e}). Using hardcoded key — replace sk-YOUR-KEY-HERE")

client = OpenAI(api_key=OPENAI_API_KEY)

# ── Model config (matches paper) ──────────────────────────────────────────────
MODEL = "gpt-4o-mini"
TEMPERATURE = 0.3
TOP_P = 0.7

print(f"✅ Client ready — model: {MODEL}")

## Cell 3: Exact Prompts from prompts.py
These are the **exact system prompts** from the official MAMMQA repository.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# STAGE 1 PROMPT — Modality Expert Agent
# Task: identify modality, extract insights, break into sub-questions
# Does NOT answer the question directly
# ─────────────────────────────────────────────────────────────────────────────
Agent_stage1_system_prompt = """
You are an expert agent specialized in analyzing single-modality inputs (such as text, table, or image) and answering questions by extracting insights and systematically breaking down complex questions into simpler subquestions.

### **Task**:
You will be provided with an input and a related question. Your job is to:

Step 1:  
- Identify the modality type of the provided input.  
  Possible types: text(s), table(s), or image(s).

Step 2:  
- Clearly understand the question and carefully analyze the input.  
- Extract insights relevant to the question, example:
  - Key information (numbers, statistics, entities, trends).
  - Temporal insights (time, date, durations, timelines, etc.).

Examples for Step 2:  
- Text example insight:  
  "The text mentions that sales increased by 20 percent from January to March, highlighting quarterly growth."  
- Table example insight:  
  "The table shows the peak attendance (350 people) occurred on Saturday, June 12, 2023, indicating highest weekend engagement."  
- Image example insight:  
  "The image clearly indicates a street sign labeled '5th Avenue' and a clock showing the time as 2:15 PM, suggesting the photo was taken in mid-afternoon."

Step 3:  
- Based on these extracted insights, carefully break down the main question into simpler and more direct subquestions or counter-questions.

Examples for Step 3:  
- Main Question: "What was the monthly growth rate during Q1?"  
  - Subquestions:  
    - "What were the sales figures for January, February, and March individually?"  
    - "By how much did the sales figures change each month?"  

- Main Question: "When was attendance lowest and highest during the event period?"  
  - Subquestions:  
    - "Which date had the lowest attendance according to the provided table?"  
    - "Which date had the highest attendance according to the provided table?"

- Main Question: "At what time was the image captured?"  
  - Subquestion:  
    - "What specific time details are visible in the image?"

    
## **Important Additional Guidelines & Formatting**:

- Always think step-by-step through your analysis.
- Clearly output the identified modality type in:
  <modality> identified modality type here </modality>

- Clearly output your extracted insights in:
  <insights> your extracted insights here </insights>

- If possible, provide the final answer to original question within:
  <answer> your final answer here </answer>

- Provide answers to subquestions, wrap these in:
  <subanswer> your answer to subquestion here </subanswer>

- Only use the provided data.  
  Do not include any external or internal knowledge beyond what's explicitly given.
"""

# ─────────────────────────────────────────────────────────────────────────────
# STAGE 2 PROMPT — Cross-Modal Synthesis Agent
# Anchored on ONE modality's insight + raw data from other two
# ─────────────────────────────────────────────────────────────────────────────
Agent_stage2_system_prompt = """
You are an expert cross agent specialized in analyzing multiple-modalities (such as text, table, or image), insights from specialised agent(s) (such as text, table, or image) and answering questions by extracting insights and systematically breaking down complex questions into simpler subquestions.

### **Task**:
You will be provided with multiple inputs ( insights from a specialised agent(s) and multimodal input(s) ) and a related question. Your job is to:

Step 1:  
- Clearly understand the question and carefully analyze the input.  
- Extract insights relevant to the question, example:
  - Key information (numbers, statistics, entities, trends).
  - Temporal insights (time, date, durations, timelines, etc.).

Step 2:  
- Based on these extracted insights and agent insights, carefully break down the main question into simpler and more direct subquestions or counter-questions.

    
## **Important Additional Guidelines & Formatting**:

- Always think step-by-step through your analysis.

- Clearly output your extracted insights in:
  <insights> your extracted insights here </insights>

- Provide answers to subquestions, wrap these in:
  <subanswer> your answer to subquestion here </subanswer>

- Provide the final answer to original question within:
  <answer> your final answer here </answer>


- Only use the provided data.  
  Do not include any external or internal knowledge beyond what's explicitly given.
"""

# ─────────────────────────────────────────────────────────────────────────────
# STAGE 3 PROMPT — Aggregator Agent
# Blind to raw inputs — sees only Stage 2 outputs
# Resolves conflicts, produces final grounded answer
# ─────────────────────────────────────────────────────────────────────────────
Agent_stage3_system_prompt = """
You are the final aggregator agent. Your input consists of three responses generated by cross-modal synthesis agents. Each response results from combining one modality's reasoning with the evidence from the other two modalities for the given question. Your task is to generate the most accurate final answer by following these rules:

(A) Consistency Check:

If at least two responses provide the same answer along with clear, robust reasoning, select that answer as final.

(B) Fallback Rule:

If two responses indicate that the available information is insufficient but one response gives a concrete answer with detailed evidence, choose the concrete answer.

(C) Conflict Resolution:

If all three responses differ, examine the quality of their reasoning. Weigh the clarity, depth, and coherence of the explanations, and select the answer with the strongest supporting rationale.

(D) Final Synthesis:

Provide your final answer along with a brief explanation summarizing the key points that influenced your decision.

Ensure that your decision-making is transparent, logically consistent."""

print("✅ All 3 stage prompts loaded")

## Cell 4: Agent Functions (from agents.py)
Exact implementation from the official MAMMQA repository.

In [ ]:
def process_image(base_list):
    """Convert image URLs to OpenAI vision format."""
    images = []
    for base in base_list:
        if base == "no_image" or not base:
            continue
        try:
            images.append({"type": "image_url", "image_url": {"url": base}})
        except Exception:
            continue
    return images


# ─── STAGE 1 AGENTS ──────────────────────────────────────────────────────────

def text_agent(client, question, texts, model=MODEL):
    """Stage 1: Extract insights from TEXT modality only."""
    messages = [{"role": "system", "content": Agent_stage1_system_prompt}]
    messages.append({
        "role": "user",
        "content": f"Here is the text data:\n{texts}\n Here is the question: {question}"
    })
    response = client.chat.completions.create(
        model=model, messages=messages, temperature=TEMPERATURE, top_p=TOP_P
    )
    return response.choices[0].message.content


def table_agent(client, question, tables, model=MODEL):
    """Stage 1: Extract insights from TABLE modality only."""
    messages = [{"role": "system", "content": Agent_stage1_system_prompt}]
    messages.append({
        "role": "user",
        "content": f"Here is the markdown table data:\n{tables}\n Here is the question: {question}"
    })
    response = client.chat.completions.create(
        model=model, messages=messages, temperature=TEMPERATURE, top_p=TOP_P
    )
    return response.choices[0].message.content


def image_agent(client, question, image_urls, model=MODEL):
    """Stage 1: Extract insights from IMAGE modality only."""
    messages = [{"role": "system", "content": Agent_stage1_system_prompt}]
    content = [{"type": "text", "text": f"Here is the question: {question}"}]
    content += process_image(image_urls)
    messages.append({"role": "user", "content": content})
    response = client.chat.completions.create(
        model=model, messages=messages, temperature=TEMPERATURE, top_p=TOP_P
    )
    return response.choices[0].message.content


# ─── STAGE 2 AGENTS ──────────────────────────────────────────────────────────

def text_cross_agent(client, question, text_insight, tables, images, model=MODEL):
    """Stage 2: Anchored on text insight, uses raw table + image."""
    messages = [{"role": "system", "content": Agent_stage2_system_prompt}]
    content = [{
        "type": "text",
        "text": f"Here is the text insight:\n{text_insight}\n Here is the Question: {question}\n Here is the markdown table:\n{tables}"
    }]
    content += process_image(images)
    messages.append({"role": "user", "content": content})
    response = client.chat.completions.create(
        model=model, messages=messages, temperature=TEMPERATURE, top_p=TOP_P
    )
    return response.choices[0].message.content


def table_cross_agent(client, question, text, table_insight, images, model=MODEL):
    """Stage 2: Anchored on table insight, uses raw text + image."""
    messages = [{"role": "system", "content": Agent_stage2_system_prompt}]
    content = [{
        "type": "text",
        "text": f"Here is the table insights:\n{table_insight}\n\n Here is the Question: {question}\n Here is the text data:\n{text}"
    }]
    content += process_image(images)
    messages.append({"role": "user", "content": content})
    response = client.chat.completions.create(
        model=model, messages=messages, temperature=TEMPERATURE, top_p=TOP_P
    )
    return response.choices[0].message.content


def image_cross_agent(client, question, text, tables, image_insight, model=MODEL):
    """Stage 2: Anchored on image insight, uses raw text + table."""
    messages = [{"role": "system", "content": Agent_stage2_system_prompt}]
    user_message = {
        "role": "user",
        "content": (
            f"Here is the image insights:\n{image_insight}\n"
            f"Here is the Question: {question}\n"
            f"Here is the text data:\n{text}\n"
            f"Here is the markdown table:\n{tables}\n"
        )
    }
    messages.append(user_message)
    response = client.chat.completions.create(
        model=model, messages=messages, temperature=TEMPERATURE, top_p=TOP_P
    )
    return response.choices[0].message.content


# ─── STAGE 3 AGENT ───────────────────────────────────────────────────────────

def reasoning_agent(client, question, text_cross_output, table_cross_output, image_cross_output, model=MODEL):
    """Stage 3: Aggregator — blind to raw inputs, resolves conflicts."""
    messages = [{"role": "system", "content": Agent_stage3_system_prompt}]
    content = [{
        "type": "text",
        "text": (
            f"Here is the Question: {question}\n"
            f"Here is the text cross output:\n{text_cross_output}\n"
            f"Here is the table cross output:\n{table_cross_output}\n"
            f"Here is the image cross output:\n{image_cross_output}"
        )
    }]
    messages.append({"role": "user", "content": content})
    response = client.chat.completions.create(
        model=model, messages=messages, temperature=TEMPERATURE, top_p=TOP_P
    )
    return response.choices[0].message.content


# ─── FULL MAMMQA PIPELINE ────────────────────────────────────────────────────

def get_answer_MM(client, question, text, tables, images, model=MODEL, verbose=True):
    """
    Full 3-stage MAMMQA pipeline.
    Returns dict with outputs from all 7 agents.
    """
    results = {}

    if verbose:
        print(f"\n{'='*70}")
        print(f"QUESTION: {question}")
        print(f"{'='*70}")

    # ── STAGE 1 ──────────────────────────────────────────────────────────────
    if verbose: print("\n▶ STAGE 1: Running Modality Expert Agents...")

    text_insight   = text_agent(client, question, text, model)
    table_insight  = table_agent(client, question, tables, model)
    image_insight  = image_agent(client, question, images, model)

    results['Text Agent Output']  = text_insight
    results['Table Agent Output'] = table_insight
    results['Image Agent Output'] = image_insight

    if verbose:
        print("  ✅ Text Expert Done")
        print("  ✅ Table Expert Done")
        print("  ✅ Image Expert Done")

    # ── STAGE 2 ──────────────────────────────────────────────────────────────
    if verbose: print("\n▶ STAGE 2: Running Cross-Modal Synthesis Agents...")

    text_cross  = text_cross_agent(client, question, text_insight, tables, images, model)
    table_cross = table_cross_agent(client, question, text, table_insight, images, model)
    image_cross = image_cross_agent(client, question, text, tables, image_insight, model)

    results['Text Cross Agent Output']  = text_cross
    results['Table Cross Agent Output'] = table_cross
    results['Image Cross Agent Output'] = image_cross

    if verbose:
        print("  ✅ Text-Anchored Synthesis Done")
        print("  ✅ Table-Anchored Synthesis Done")
        print("  ✅ Image-Anchored Synthesis Done")

    # ── STAGE 3 ──────────────────────────────────────────────────────────────
    if verbose: print("\n▶ STAGE 3: Running Aggregator Agent...")

    final_answer = reasoning_agent(client, question, text_cross, table_cross, image_cross, model)
    results['Final Answer'] = final_answer

    if verbose:
        print("  ✅ Aggregator Done")

    return results


print("✅ All agent functions defined")

## Cell 5: Download MultimodalQA Dataset Files

In [ ]:
DATA_DIR = "/kaggle/working/mmqa_data"
os.makedirs(DATA_DIR, exist_ok=True)

BASE_URL = "https://multimodalqa-images.s3-us-west-2.amazonaws.com/final_dataset_release"

FILES = {
    "dev":    (f"{BASE_URL}/MMQA_dev.jsonl.gz",    f"{DATA_DIR}/MMQA_dev.jsonl"),
    "texts":  (f"{BASE_URL}/MMQA_texts.jsonl.gz",  f"{DATA_DIR}/MMQA_texts.jsonl"),
    "tables": (f"{BASE_URL}/MMQA_tables.jsonl.gz", f"{DATA_DIR}/MMQA_tables.jsonl"),
    "images": (f"{BASE_URL}/MMQA_images.jsonl.gz", f"{DATA_DIR}/MMQA_images.jsonl"),
}

def download_gz(url, dest):
    gz = dest + ".gz"
    if not os.path.exists(dest):
        print(f"  ⏬ Downloading {os.path.basename(dest)}...")
        urllib.request.urlretrieve(url, gz)
        with gzip.open(gz, 'rb') as fin, open(dest, 'wb') as fout:
            shutil.copyfileobj(fin, fout)
        os.remove(gz)
        print(f"  ✅ Saved to {dest}")
    else:
        print(f"  ✅ {os.path.basename(dest)} already exists")

print("Downloading MultimodalQA dataset...")
for name, (url, dest) in FILES.items():
    download_gz(url, dest)

print("\n✅ All files ready")

## Cell 6: Load and Index the Dataset (from Dataloader.py)

In [ ]:
def load_jsonl(path):
    data = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    data.append(json.loads(line))
                except json.JSONDecodeError:
                    pass
    return data


def make_columns_unique(df):
    new_columns, counts = [], {}
    for col in df.columns:
        if col in counts:
            counts[col] += 1
            new_columns.append(f"{col}_{counts[col]}")
        else:
            counts[col] = 0
            new_columns.append(col)
    df.columns = new_columns
    return df


def parse_table(data):
    title = data.get('title', 'No Title')
    table_block = data.get('table', {})
    headers = [h.get('column_name', '') for h in table_block.get('header', [])]
    rows = table_block.get('table_rows', [])
    formatted_rows = [[cell.get('text', '') for cell in row] for row in rows]
    if formatted_rows:
        if headers and len(headers) == len(formatted_rows[0]):
            df = pd.DataFrame(formatted_rows, columns=headers)
        else:
            df = pd.DataFrame(formatted_rows)
        df = make_columns_unique(df)
    else:
        df = pd.DataFrame()
    return {"title": title, "markdown_table": df.to_markdown(index=False) if not df.empty else ""}


def parse_text(data):
    return {"title": data.get('title', 'No Title'), "text": data.get('text', '')}


def parse_image(data):
    return {"title": data.get('title', 'No Title'), "path": data.get('path', '')}


print("Loading dataset into memory (this takes ~30 seconds)...")
dev_data    = load_jsonl(f"{DATA_DIR}/MMQA_dev.jsonl")
texts_raw   = load_jsonl(f"{DATA_DIR}/MMQA_texts.jsonl")
tables_raw  = load_jsonl(f"{DATA_DIR}/MMQA_tables.jsonl")
images_raw  = load_jsonl(f"{DATA_DIR}/MMQA_images.jsonl")

# Build lookup indices
table_lookup = {e['id']: parse_table(e) for e in tables_raw if e.get('id')}
text_lookup  = {e['id']: parse_text(e)  for e in texts_raw  if e.get('id')}
image_lookup = {e['id']: parse_image(e) for e in images_raw if e.get('id')}

print(f"\n✅ Loaded {len(dev_data):,} dev questions")
print(f"✅ Indexed {len(text_lookup):,} text passages")
print(f"✅ Indexed {len(table_lookup):,} tables")
print(f"✅ Indexed {len(image_lookup):,} images")

## Cell 7: Build Unified Examples
We assemble question + text + table + image for each example, exactly like the Dataloader does.

In [ ]:
WIKI_IMAGE_API = "https://en.wikipedia.org/w/api.php"

def get_wikipedia_image_url(title):
    """Fetch the main image URL for a Wikipedia article title."""
    try:
        params = {
            "action": "query", "titles": title, "prop": "pageimages",
            "pithumbsize": 500, "format": "json"
        }
        r = requests.get(WIKI_IMAGE_API, params=params, timeout=5)
        pages = r.json().get("query", {}).get("pages", {})
        for page in pages.values():
            thumb = page.get("thumbnail", {}).get("source")
            if thumb:
                return thumb
    except Exception:
        pass
    return None


def build_example(entry):
    """Build a complete multimodal example from a dev_data entry."""
    meta      = entry.get('metadata', {})
    table_id  = meta.get('table_id')
    text_ids  = meta.get('text_doc_ids', [])
    image_ids = meta.get('image_doc_ids', [])

    # ── Text: join all passages
    texts_data = [text_lookup[tid] for tid in text_ids if tid in text_lookup]
    combined_text = "\n".join(
        f"title: {t['title']}, text: {t['text']}" for t in texts_data
    ) or "No text available"

    # ── Table: markdown
    table_data = table_lookup.get(table_id)
    table_md = table_data['markdown_table'] if table_data else "No table data"

    # ── Images: try Wikipedia API, fall back to title-only placeholder
    image_urls = []
    for iid in image_ids:
        img = image_lookup.get(iid)
        if img:
            url = get_wikipedia_image_url(img['title'])
            if url:
                image_urls.append(url)
            else:
                # Placeholder text so image agent still gets context
                image_urls.append(f"[Image: {img['title']}]")

    return {
        "question":    entry.get('question', ''),
        "answer":      entry.get('answers', {}).get('answer', ''),
        "type":        meta.get('type', 'unknown'),
        "modalities":  meta.get('modalities', []),
        "text":        combined_text,
        "table":       table_md,
        "images":      image_urls,
    }


# ── Select 5 examples — one of each type ─────────────────────────────────────
# Types: TextQ, TableQ, ImageQ, Compose (cross-modal)

target_types = ['TextQ', 'TableQ', 'ImageQ', 'Compose']
selected, seen_types = [], set()

for entry in dev_data:
    q_type = entry.get('metadata', {}).get('type', '')
    if q_type in target_types and q_type not in seen_types:
        seen_types.add(q_type)
        selected.append(entry)
    if len(seen_types) == len(target_types):
        break

# Add one more Compose for variety
for entry in dev_data:
    if entry.get('metadata', {}).get('type') == 'Compose' and entry not in selected:
        selected.append(entry)
        break

print(f"Selected {len(selected)} examples: {[e['metadata']['type'] for e in selected]}")
print("\nBuilding examples (fetching Wikipedia images)...")

examples = [build_example(e) for e in selected]

for i, ex in enumerate(examples):
    print(f"\nExample {i+1} [{ex['type']}]:")
    print(f"  Q: {ex['question'][:80]}..." if len(ex['question']) > 80 else f"  Q: {ex['question']}")
    print(f"  A: {ex['answer']}")
    print(f"  Images found: {len([u for u in ex['images'] if not u.startswith('[Image')])}")

print("\n✅ All examples ready")

## Cell 8: Run the Full MAMMQA Pipeline
This cell runs all 7 agents on each example and stores every intermediate output.

In [ ]:
all_results = []

for i, ex in enumerate(examples):
    print(f"\n{'#'*70}")
    print(f"  EXAMPLE {i+1}/{len(examples)}  |  Type: {ex['type']}")
    print(f"{'#'*70}")

    result = get_answer_MM(
        client   = client,
        question = ex['question'],
        text     = ex['text'],
        tables   = ex['table'],
        images   = ex['images'],
        model    = MODEL,
        verbose  = True
    )

    result['question']   = ex['question']
    result['true_answer'] = ex['answer']
    result['type']        = ex['type']
    all_results.append(result)

    print(f"\n  ✅ Example {i+1} complete")

print(f"\n\n{'='*70}")
print(f"  ALL {len(all_results)} EXAMPLES PROCESSED")
print(f"{'='*70}")

## Cell 9: Display All Intermediate Outputs
See exactly what each of the 7 agents produced.

In [ ]:
def extract_answer_tag(text):
    """Pull text between <answer>...</answer> tags if present."""
    m = re.search(r'<answer>(.*?)</answer>', text, re.DOTALL | re.IGNORECASE)
    return m.group(1).strip() if m else text.strip()


STAGE_KEYS = [
    ('STAGE 1 — Text Expert',          'Text Agent Output'),
    ('STAGE 1 — Table Expert',         'Table Agent Output'),
    ('STAGE 1 — Image Expert',         'Image Agent Output'),
    ('STAGE 2 — Text-Anchored Synth',  'Text Cross Agent Output'),
    ('STAGE 2 — Table-Anchored Synth', 'Table Cross Agent Output'),
    ('STAGE 2 — Image-Anchored Synth', 'Image Cross Agent Output'),
    ('STAGE 3 — Final Answer',         'Final Answer'),
]

for i, result in enumerate(all_results):
    print(f"\n{'='*70}")
    print(f"EXAMPLE {i+1}  |  Type: {result['type']}")
    print(f"QUESTION: {result['question']}")
    print(f"TRUE ANSWER: {result['true_answer']}")
    print(f"{'='*70}")

    for label, key in STAGE_KEYS:
        output = result.get(key, 'N/A')
        print(f"\n┌─ {label}")
        # Print first 400 chars to keep output readable
        preview = output[:400] + '...' if len(output) > 400 else output
        for line in preview.split('\n'):
            print(f"│  {line}")
        print(f"└{'─'*67}")

## Cell 10: Extract Final Answers and Evaluate

In [ ]:
def normalize(text):
    """Lowercase, strip punctuation, split to tokens."""
    return re.sub(r'\W+', ' ', str(text).lower().strip()).split()


def f1_score(pred, true):
    pred_tokens = normalize(pred)
    true_tokens = normalize(true)
    if not pred_tokens or not true_tokens:
        return 0.0
    common = Counter(pred_tokens) & Counter(true_tokens)
    n_common = sum(common.values())
    if n_common == 0:
        return 0.0
    precision = n_common / len(pred_tokens)
    recall    = n_common / len(true_tokens)
    return 2 * precision * recall / (precision + recall)


def exact_match(pred, true):
    return normalize(pred) == normalize(true)


print(f"{'─'*65}")
print(f"{'#':<3} {'Type':<10} {'True Answer':<25} {'Predicted':<25} {'EM':<5} {'F1':<5}")
print(f"{'─'*65}")

em_scores, f1_scores = [], []
rows = []

for i, result in enumerate(all_results):
    final_raw   = result.get('Final Answer', '')
    predicted   = extract_answer_tag(final_raw)
    true_answer = str(result['true_answer'])

    em = exact_match(predicted, true_answer)
    f1 = f1_score(predicted, true_answer)

    em_scores.append(em)
    f1_scores.append(f1)

    pred_short = (predicted[:22] + '...') if len(predicted) > 25 else predicted
    true_short = (true_answer[:22] + '...') if len(true_answer) > 25 else true_answer

    print(f"{i+1:<3} {result['type']:<10} {true_short:<25} {pred_short:<25} {'✅' if em else '❌':<5} {f1:.2f}")

    rows.append({
        'example':    i + 1,
        'type':       result['type'],
        'question':   result['question'],
        'true_answer': true_answer,
        'predicted':  predicted,
        'exact_match': em,
        'f1':         round(f1, 3),
    })

print(f"{'─'*65}")
print(f"{'OVERALL':<14} EM: {sum(em_scores)/len(em_scores)*100:.1f}%   F1: {sum(f1_scores)/len(f1_scores)*100:.1f}%")
print(f"{'─'*65}")

## Cell 11: Full Detail View — One Example

In [ ]:
# ── Change EXAMPLE_IDX (0-4) to inspect a different example ──────────────────
EXAMPLE_IDX = 3   # Compose question is the most interesting to examine

result = all_results[EXAMPLE_IDX]
ex     = examples[EXAMPLE_IDX]

print(f"FULL PIPELINE TRACE — Example {EXAMPLE_IDX+1}")
print(f"{'='*70}")
print(f"Question : {result['question']}")
print(f"Type     : {result['type']}")
print(f"Modalities: {ex['modalities']}")
print(f"True Answer: {result['true_answer']}")

print(f"\n{'─'*70}")
print("INPUT DATA SUMMARY")
print(f"{'─'*70}")
print(f"Text (first 300 chars):\n  {ex['text'][:300]}...")
print(f"\nTable (first 500 chars):\n  {ex['table'][:500]}..." if ex['table'] != 'No table data' else "\nTable: No table data")
print(f"\nImage URLs: {ex['images']}")

for label, key in STAGE_KEYS:
    print(f"\n{'='*70}")
    print(f"{label}")
    print(f"{'='*70}")
    print(result.get(key, 'N/A'))

## Cell 12: Save Results to CSV

In [ ]:
import csv

OUTPUT_CSV = "/kaggle/working/mammqa_results.csv"

# Build flat rows for CSV
csv_rows = []
for i, result in enumerate(all_results):
    row = {
        'example':            i + 1,
        'type':               result['type'],
        'question':           result['question'],
        'true_answer':        result['true_answer'],
        'text_agent':         result.get('Text Agent Output', ''),
        'table_agent':        result.get('Table Agent Output', ''),
        'image_agent':        result.get('Image Agent Output', ''),
        'text_cross':         result.get('Text Cross Agent Output', ''),
        'table_cross':        result.get('Table Cross Agent Output', ''),
        'image_cross':        result.get('Image Cross Agent Output', ''),
        'final_answer_raw':   result.get('Final Answer', ''),
        'predicted_answer':   rows[i]['predicted'],
        'exact_match':        rows[i]['exact_match'],
        'f1':                 rows[i]['f1'],
    }
    csv_rows.append(row)

df_results = pd.DataFrame(csv_rows)
df_results.to_csv(OUTPUT_CSV, index=False)

print(f"✅ Results saved to: {OUTPUT_CSV}")
print(f"   Shape: {df_results.shape}")
print("\nPreview:")
df_results[['example','type','question','true_answer','predicted_answer','exact_match','f1']].head(10)

## Cell 13: Quick Experiment — Compare MAMMQA vs Plain CoT
Run the CoT baseline on the same examples to see the difference.

In [ ]:
def get_answer_cot(client, question, text, table, images, model=MODEL):
    """CoT baseline — single model, all modalities at once."""
    system_message = {
        "role": "system",
        "content": (
            "You are provided with specific data in the form of text, tables, and images.\n"
            "Answer the question using ONLY the provided data.\n"
            "Output format:\n"
            "<reasoning>your step-by-step reasoning here</reasoning>\n"
            "<answer>your final answer here</answer>\n"
            "Let's think step by step."
        )
    }
    content = [{
        "type": "text",
        "text": f"Question: {question}\nText:\n{text}\nTable:\n{table}"
    }]
    content += process_image(images)
    messages = [system_message, {"role": "user", "content": content}]
    ans = client.chat.completions.create(
        model=model, messages=messages, temperature=TEMPERATURE, top_p=TOP_P
    )
    return ans.choices[0].message.content


print("Running CoT baseline on same examples...\n")

cot_em, cot_f1 = [], []
print(f"{'─'*65}")
print(f"{'#':<3} {'Type':<10} {'True Answer':<22} {'MAMMQA':<6} {'CoT':<6}")
print(f"{'─'*65}")

for i, ex in enumerate(examples):
    cot_raw  = get_answer_cot(client, ex['question'], ex['text'], ex['table'], ex['images'])
    cot_pred = extract_answer_tag(cot_raw)
    true_ans = str(ex['answer'])

    cot_em.append(exact_match(cot_pred, true_ans))
    cot_f1.append(f1_score(cot_pred, true_ans))

    mammqa_em_sym = '✅' if em_scores[i] else '❌'
    cot_em_sym    = '✅' if cot_em[-1]  else '❌'

    true_short = (true_ans[:19] + '...') if len(true_ans) > 22 else true_ans
    print(f"{i+1:<3} {ex['type']:<10} {true_short:<22} {mammqa_em_sym:<6} {cot_em_sym:<6}")

print(f"{'─'*65}")
print(f"{'OVERALL EM':<14} MAMMQA: {sum(em_scores)/len(em_scores)*100:.1f}%   CoT: {sum(cot_em)/len(cot_em)*100:.1f}%")
print(f"{'OVERALL F1':<14} MAMMQA: {sum(f1_scores)/len(f1_scores)*100:.1f}%   CoT: {sum(cot_f1)/len(cot_f1)*100:.1f}%")
print(f"{'─'*65}")

## Cell 14: Run on Your Own Custom Question
Test MAMMQA on any question you want by providing your own text, table, and image.

In [ ]:
# ── Modify these inputs to run on your own question ───────────────────────────

MY_QUESTION = "Which city hosted the first summer Olympics in the modern era?"

MY_TEXT = """
title: Summer Olympics History
text: The modern Olympic Games were revived in 1896 and held in Athens, Greece.
They were organized by Pierre de Coubertin and attended by 14 nations.
The 1900 Olympics were held in Paris, France as part of the World's Fair.
The 1904 Olympics were the first held in the United States, in St. Louis.
"""

MY_TABLE = """
| Year | City         | Country       | Nations |
|------|--------------|---------------|--------|
| 1896 | Athens       | Greece        | 14     |
| 1900 | Paris        | France        | 24     |
| 1904 | St. Louis    | United States | 12     |
| 1908 | London       | Great Britain | 22     |
| 1912 | Stockholm    | Sweden        | 28     |
"""

MY_IMAGES = []  # Leave empty or add image URLs

# ─────────────────────────────────────────────────────────────────────────────
print("Running MAMMQA on custom question...")
my_result = get_answer_MM(
    client   = client,
    question = MY_QUESTION,
    text     = MY_TEXT,
    tables   = MY_TABLE,
    images   = MY_IMAGES,
    model    = MODEL,
    verbose  = True
)

print(f"\n{'='*70}")
print("FINAL ANSWER FROM MAMMQA:")
print(f"{'='*70}")
print(my_result['Final Answer'])

---
## Summary

### What this notebook demonstrated:

| Stage | Agent | Input | Output |
|-------|-------|-------|--------|
| 1 | Text Expert | Text passage + Question | Extracted insights + sub-answers |
| 1 | Table Expert | Markdown table + Question | Extracted insights + sub-answers |
| 1 | Image Expert | Image URL(s) + Question | Extracted visual insights |
| 2 | Text-Cross | Text insight + raw table + raw image | Candidate answer (text-anchored) |
| 2 | Table-Cross | Table insight + raw text + raw image | Candidate answer (table-anchored) |
| 2 | Image-Cross | Image insight + raw text + raw table | Candidate answer (image-anchored) |
| 3 | Aggregator | 3 candidate answers (blind to raw inputs) | Final grounded answer |

### Key design choices in the actual code:
- Stage 1 uses `Agent_stage1_system_prompt` — focuses on extraction, NOT answering
- Stage 2 uses `Agent_stage2_system_prompt` — cross-modal synthesis from one anchor
- Stage 3 uses `Agent_stage3_system_prompt` — aggregation rules (A)-(D)
- The aggregator receives the **question** but NOT the raw text/table/image
- Temperature = 0.3, Top-p = 0.7 (as per paper)

### Next steps for research:
1. Add a retrieval stage **before** Stage 1 (open-domain MAMMQA)
2. Add inter-stage feedback (Stage 2 can signal Stage 1 to re-extract)
3. Replace single agents with diverse ensembles per modality
4. Train a learned meta-aggregator to replace the hand-crafted rules in Stage 3